# Disaster Tweets Gemini API Experiment

Separate notebook for prompt-based classification using Gemini.

This notebook is isolated from the Groq notebook so the two API experiments can run independently.


## 1. Goal

Use Gemini as a zero-shot classifier for disaster tweet detection.

Planned output:
- validation `F1` on a local train/valid split
- optional Kaggle submission file for `test.csv`


## 2. Key setup

Store your API key in an environment variable or local `.env` file.

Recommended variable names:
- `GEMINI_API_KEY`
- `GEMINI_MODEL` (optional, default `gemini-2.5-flash`)

Do not hardcode the key inside the notebook or commit it to GitHub.


In [ ]:
# Install dependencies inside the notebook so Run All works without opening a terminal.
import importlib.util
import subprocess
import sys

required_packages = ['google-genai', 'python-dotenv', 'pandas', 'scikit-learn', 'tqdm']
missing_packages = []
for package in required_packages:
    module_name = package.replace('-', '_')
    if importlib.util.find_spec(module_name) is None:
        missing_packages.append(package)

if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])

from pathlib import Path
import json
import os
import re

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

try:
    from google import genai
except ImportError:
    genai = None

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data'
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'

if load_dotenv is not None:
    load_dotenv()

API_KEY = os.getenv('GEMINI_API_KEY')
GEMINI_MODEL = os.getenv('GEMINI_MODEL', 'gemini-2.5-flash')
print('API key loaded:', bool(API_KEY))
print('Gemini model:', GEMINI_MODEL)
print('google-genai available:', genai is not None)


## 3. Data loading

We reuse the same `train.csv` and `test.csv` files, but keep the Gemini workflow separate.


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH) if TEST_PATH.exists() else None

print(train_df.shape)
if test_df is not None:
    print(test_df.shape)


## 4. Lightweight text cleanup

For Gemini prompting we keep cleaning minimal so the model sees natural language.

Suggested cleanup:
- lowercasing
- remove URLs
- normalize whitespace
- limit very long tweets to reduce token usage


In [ ]:
MAX_TEXT_CHARS = 220

def light_clean(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:MAX_TEXT_CHARS]

train_df['gemini_text'] = train_df['text'].apply(light_clean)
if test_df is not None:
    test_df['gemini_text'] = test_df['text'].apply(light_clean)

train_df[['text', 'gemini_text', 'target']].head()


## 5. Prompt design

Keep the prompt strict and short so token usage stays low.

Recommended response format:
- return only comma-separated `0/1` labels
- no explanation in the final answer
- one label per tweet in the same order


In [ ]:
SYSTEM_PROMPT = 'Return only a single digit: 0 or 1.'

def build_prompt(tweet_text):
    return f'Tweet: {tweet_text}\nLabel:'


## 6. Batching strategy

Start conservatively to avoid token-limit errors.

Suggested starting point:
- batch size: `2`
- validation subset: `100`
- scale up only if the run is stable


In [ ]:
BATCH_SIZE = 1
VALIDATION_SIZE = 100
MAX_OUTPUT_TOKENS = 128
TEMPERATURE = 0.0

train_part, valid_part = train_test_split(
    train_df,
    test_size=VALIDATION_SIZE,
    random_state=42,
    stratify=train_df['target'],
)

print(train_part.shape, valid_part.shape)


## 7. API inference

This cell creates the Gemini client and defines the prediction helper.


In [ ]:
if API_KEY is None:
    raise ValueError('GEMINI_API_KEY is missing. Put it in .env and rerun the setup cell.')

if genai is None:
    raise ImportError('google-genai package is missing. Run the install cell above and rerun the imports.')

client = genai.Client(api_key=API_KEY)

def parse_label_list(raw_text, expected_len):
    labels = [int(x) for x in re.findall(r'[01]', raw_text)]
    if len(labels) < expected_len:
        raise ValueError(f'Could not parse enough labels from response: {raw_text!r}')
    return labels[:expected_len]

def extract_response_text(response):
    text = getattr(response, 'text', None)
    if text:
        return text.strip()
    parts = []
    for candidate in getattr(response, 'candidates', []) or []:
        content = getattr(candidate, 'content', None)
        for part in getattr(content, 'parts', []) or []:
            part_text = getattr(part, 'text', None)
            if part_text:
                parts.append(part_text)
    return ' '.join(parts).strip()

def _call_llm(tweet_texts):
    if len(tweet_texts) != 1:
        raise ValueError('This stable mode expects exactly one tweet per request.')
    prompt = build_prompt(tweet_texts[0])
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config={
            'system_instruction': SYSTEM_PROMPT,
            'temperature': TEMPERATURE,
            'max_output_tokens': MAX_OUTPUT_TOKENS,
        },
    )
    raw_text = extract_response_text(response)
    if not raw_text:
        feedback = getattr(response, 'prompt_feedback', None)
        raise ValueError(f'Gemini returned an empty response body. prompt_feedback={feedback!r}, response={response!r}')
    return parse_label_list(raw_text, 1)

def predict_batch_with_llm(tweet_texts):
    if not tweet_texts:
        return []
    predictions = []
    for text in tweet_texts:
        predictions.extend(_call_llm([text]))
    return predictions


## 8. Run validation and score

The final cell runs the Gemini classifier on the local validation split and prints `F1`.


In [ ]:
all_predictions = []
texts = valid_part['gemini_text'].tolist()
for text in texts:
    all_predictions.extend(predict_batch_with_llm([text]))

valid_predictions = pd.Series(all_predictions, index=valid_part.index)
valid_f1 = f1_score(valid_part['target'], valid_predictions)
print(f'Validation F1: {valid_f1:.4f}')
print(classification_report(valid_part['target'], valid_predictions))
print('Confusion matrix:')
print(confusion_matrix(valid_part['target'], valid_predictions))
